# Phase 4, Stage 4: Compilation and Between-Band Comparison

This notebook pulls together the outputs of Stages 1-3 into a small number of clean summary tables, and builds the **between-band comparison** (Stage 4.7 in `../CLAUDE.md`) that directly addresses the "disproportionate effect across rating bands" part of the SRQ.

We load:
- `descriptive_stats.csv` (Stage 1)
- `ks_test_results.csv` (Stage 3a)
- `chi_squared_results.csv` (Stage 3b)
- `ttest_cohens_d_results.csv` (Stage 3c)
- `pearson_correlation_results.csv` (Stage 3d)

and produce:
- `within_band_summary.csv` — one row per (rating band, transition), combining KS D and Cohen's d side by side
- `between_band_comparison.csv` — KS D and Cohen's d laid out with rating band as rows and transition as columns, for direct cross-band comparison
- `band_level_summary.csv` — one row per rating band, combining chi-squared/Cramer's V and Pearson's r

In [ ]:
import pandas as pd

RATING_BAND_LABELS = {
    1: 'Novice (<1000)',
    2: 'Intermediate (1000-1499)',
    3: 'Club Player (1500-1999)',
    4: 'Advanced (2000-2299)',
    5: 'Expert/Master (2300+)',
}
BAND_ORDER = [RATING_BAND_LABELS[b] for b in sorted(RATING_BAND_LABELS)]
TRANSITION_ORDER = ['Bin 1 -> Bin 2', 'Bin 2 -> Bin 3', 'Bin 3 -> Bin 4']

descriptive = pd.read_csv('../results/descriptive_stats.csv')
ks = pd.read_csv('../results/ks_test_results.csv')
chi2 = pd.read_csv('../results/chi_squared_results.csv')
ttest = pd.read_csv('../results/ttest_cohens_d_results.csv')
pearson = pd.read_csv('../results/pearson_correlation_results.csv')

print('Loaded:', len(descriptive), 'descriptive rows,', len(ks), 'KS rows,',
      len(chi2), 'chi2 rows,', len(ttest), 'ttest rows,', len(pearson), 'pearson rows')

## Within-band summary table

One row per (rating band, transition), with the KS D-statistic and Cohen's d side by side, plus their significance flags. This is the table that supports the within-band claims ("does the distribution/mean shift between adjacent pressure bins for *this* band?").

In [ ]:
within_band = ks[['rating_band', 'rating_band_label', 'transition', 'D', 'p_value', 'significant_bonferroni']].rename(
    columns={'D': 'ks_D', 'p_value': 'ks_p', 'significant_bonferroni': 'ks_significant'}
).merge(
    ttest[['rating_band', 'transition', 'mean_diff', 'cohens_d', 'p_value', 'significant_bonferroni']].rename(
        columns={'p_value': 'ttest_p', 'significant_bonferroni': 'ttest_significant'}
    ),
    on=['rating_band', 'transition']
)

within_band['rating_band_label'] = pd.Categorical(within_band['rating_band_label'], categories=BAND_ORDER, ordered=True)
within_band['transition'] = pd.Categorical(within_band['transition'], categories=TRANSITION_ORDER, ordered=True)
within_band = within_band.sort_values(['rating_band', 'transition']).reset_index(drop=True)

within_band.to_csv('../results/within_band_summary.csv', index=False)
print('Saved to ../results/within_band_summary.csv')
within_band

## Between-band comparison (Stage 4.7)

Now we pivot so rating bands become rows and transitions become columns, for both KS D and Cohen's d. This is the table that lets us read across a row (how does this band's response to pressure change across transitions?) and down a column (how does the response to *this* transition change across rating bands?) — directly addressing the "interaction" framing of the SRQ.

In [ ]:
ks_pivot = within_band.pivot(index='rating_band_label', columns='transition', values='ks_D')
ks_pivot = ks_pivot.reindex(index=BAND_ORDER, columns=TRANSITION_ORDER)
ks_pivot.columns = pd.MultiIndex.from_product([['KS_D'], ks_pivot.columns])

d_pivot = within_band.pivot(index='rating_band_label', columns='transition', values='cohens_d')
d_pivot = d_pivot.reindex(index=BAND_ORDER, columns=TRANSITION_ORDER)
d_pivot.columns = pd.MultiIndex.from_product([['cohens_d'], d_pivot.columns])

between_band = pd.concat([ks_pivot, d_pivot], axis=1)
between_band.to_csv('../results/between_band_comparison.csv')
print('Saved to ../results/between_band_comparison.csv')
between_band

## Band-level summary: chi-squared/Cramer's V and Pearson's r

These two tests don't have a "transition" dimension — they're one value per rating band already. Combining them gives a compact overview row per band.

In [ ]:
band_level = chi2[['rating_band', 'rating_band_label', 'n', 'cramers_v', 'p_value', 'significant_bonferroni']].rename(
    columns={'p_value': 'chi2_p', 'significant_bonferroni': 'chi2_significant'}
).merge(
    pearson[['rating_band', 'pearson_r', 'r_squared', 'p_value', 'significant_bonferroni']].rename(
        columns={'p_value': 'pearson_p', 'significant_bonferroni': 'pearson_significant'}
    ),
    on='rating_band'
)

band_level['rating_band_label'] = pd.Categorical(band_level['rating_band_label'], categories=BAND_ORDER, ordered=True)
band_level = band_level.sort_values('rating_band').reset_index(drop=True)

band_level.to_csv('../results/band_level_summary.csv', index=False)
print('Saved to ../results/band_level_summary.csv')
band_level